# 03 - H&M candidate union and reranking

Reranker học cách sắp xếp lại candidate từ retrieval, popularity, repeat và content similarity.

## 1. Load data and frozen retrieval model

In [ ]:
from pathlib import Path
import json
import os
import random

import numpy as np
import polars as pl
import torch
from torch import nn

In [ ]:
PROFILE = os.getenv("HM_PROFILE", "quick").lower()
MODALITY = os.getenv("HM_MODALITY", "multimodal").lower()
assert PROFILE in {"quick", "full"}
assert MODALITY in {"id", "text", "image", "multimodal"}

DATA_DIR = Path(os.getenv("HM_DATA_DIR", os.getenv("HM_WORK_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled")))
RETRIEVAL_DIR = Path(os.getenv("HM_RETRIEVAL_DIR", "/kaggle/working/hm_retrieval"))
OUTPUT_DIR = Path(os.getenv("HM_RERANK_DIR", "/kaggle/working/hm_reranking"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = json.loads((DATA_DIR / "run_config.json").read_text())
SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Profile: {PROFILE} | Modality: {MODALITY} | Device: {device}")

In [ ]:
items = pl.read_parquet(DATA_DIR / "items.parquet")
item_ids = items["item_id"].to_list()
vocab = {item_id: idx + 1 for idx, item_id in enumerate(item_ids)}

text_raw = np.load(DATA_DIR / "text_embeddings.npy").astype("float32")
image_raw = np.load(DATA_DIR / "image_embeddings.npy").astype("float32")
text = np.vstack([np.zeros((1, text_raw.shape[1]), dtype="float32"), text_raw])
image = np.vstack([np.zeros((1, image_raw.shape[1]), dtype="float32"), image_raw])

def load_history(name):
    data = pl.read_parquet(DATA_DIR / f"{name}.parquet")
    data = data.with_columns(pl.col("article_id").replace_strict(vocab, default=0).alias("idx"))
    data = data.filter(pl.col("idx") > 0).sort(["customer_id", "t_dat"])
    return {row["customer_id"]: row["idx"] for row in data.group_by("customer_id").agg(pl.col("idx")).to_dicts()}

train = load_history("train")
selection = load_history("rerank_train")
valid = load_history("valid")
test = load_history("test")
print(f"Catalog: {len(item_ids):,} | Test users: {len(test):,}")

In [ ]:
MAXLEN = 30
D_MODEL = 128

class Tower(nn.Module):
    def __init__(self):
        super().__init__()
        self.id = nn.Embedding(len(item_ids) + 1, D_MODEL, padding_idx=0)
        self.text_proj = nn.Linear(text.shape[1], D_MODEL, bias=False)
        self.image_proj = nn.Linear(image.shape[1], D_MODEL, bias=False)
        self.image_gate = nn.Parameter(torch.tensor(0.0))
        self.id_gate = nn.Parameter(torch.tensor(-2.0))
        self.register_buffer("text", torch.tensor(text))
        self.register_buffer("image", torch.tensor(image))

    def item_table(self):
        id_vectors = torch.sigmoid(self.id_gate) * self.id.weight
        if MODALITY == "id":
            vectors = id_vectors
        elif MODALITY == "text":
            vectors = self.text_proj(self.text)
        elif MODALITY == "image":
            vectors = self.image_proj(self.image)
        else:
            vectors = id_vectors + self.text_proj(self.text) + torch.sigmoid(self.image_gate) * self.image_proj(self.image)
        return nn.functional.normalize(vectors, dim=1)

    def query(self, history, table):
        mask = history.ne(0)
        weights = torch.arange(1, history.shape[1] + 1, device=history.device)[None] * mask
        query = (table[history] * weights[:, :, None]).sum(1) / weights.sum(1, keepdim=True).clamp_min(1)
        return nn.functional.normalize(query, dim=1)

retrieval_state = torch.load(RETRIEVAL_DIR / "best_retrieval.pt", map_location=device)
model = Tower().to(device)
model.load_state_dict(retrieval_state["state"])
model.eval()

## 2. Candidate union

Mỗi user nhận union từ tower, popularity, lịch sử mua lại và content KNN. Ground truth không bao giờ được chèn vào candidate.

In [ ]:
freq = np.ones(len(item_ids) + 1, dtype="float64")
freq[0] = 0
for history in train.values():
    freq[np.asarray(history)] += 1
popular_ranked = [int(idx) for idx in np.argsort(-freq) if idx != 0]
train_item_set = {item for history in train.values() for item in history}

def l2_normalize(values):
    return values / np.linalg.norm(values, axis=1, keepdims=True).clip(1e-8)

content_vectors = l2_normalize(l2_normalize(text_raw) + l2_normalize(image_raw))
TOWER_K, POPULAR_K, REPEAT_K, CONTENT_K, CANDIDATE_K = 500, 300, 100, 200, 1000
print(f"Candidate cap: {CANDIDATE_K} | Popular top item id: {popular_ranked[0]}")

In [ ]:
def unique_recent(history, limit):
    return list(dict.fromkeys(reversed(history)))[:limit]

@torch.no_grad()
def build_groups(context, targets):
    groups = []
    table = model.item_table()

    for user, truth in targets.items():
        history = context.get(user, [])[-MAXLEN:]
        source_lists = {"tower": [], "popular": popular_ranked[:POPULAR_K], "repeat": unique_recent(history, REPEAT_K), "content": []}
        tower_scores = None

        if history:
            batch = torch.tensor([[0] * (MAXLEN - len(history)) + history], device=device)
            tower_scores = (model.query(batch, table) @ table.T).squeeze()
            tower_scores[0] = -torch.inf
            source_lists["tower"] = torch.topk(tower_scores, TOWER_K).indices.cpu().tolist()
            anchor = content_vectors[np.asarray(history[-3:]) - 1].mean(0)
            content_scores = content_vectors @ (anchor / np.linalg.norm(anchor).clip(1e-8))
            source_lists["content"] = (np.argpartition(-content_scores, CONTENT_K)[:CONTENT_K] + 1).tolist()

        union, membership = [], {}
        for name, values in source_lists.items():
            for item in values:
                item = int(item)
                if item == 0:
                    continue
                membership.setdefault(item, set()).add(name)
                if item not in union and len(union) < CANDIDATE_K:
                    union.append(item)

        if tower_scores is None:
            tower_feature = np.zeros(len(union), dtype="float32")
            content_feature = np.zeros(len(union), dtype="float32")
        else:
            tower_feature = tower_scores[union].detach().cpu().numpy()
            content_feature = content_vectors[np.asarray(union) - 1] @ anchor

        features = np.column_stack([tower_feature, np.log1p(freq[union]), content_feature,
                                    [int("tower" in membership[item]) for item in union],
                                    [int("popular" in membership[item]) for item in union],
                                    [int("repeat" in membership[item]) for item in union],
                                    [int("content" in membership[item]) for item in union]]).astype("float32")
        groups.append({"user": user, "history": set(history), "truth": set(truth), "items": np.asarray(union), "x": features, "fallback": not bool(history)})
    return groups

In [ ]:
def candidate_metrics(groups, target_filter=None):
    recalls, hits, covered = [], [], set()
    for group in groups:
        truth = {item for item in group["truth"] if target_filter is None or target_filter(group, item)}
        if not truth:
            continue
        found = truth.intersection(group["items"])
        recalls.append(len(found) / len(truth))
        hits.append(float(bool(found)))
        covered.update(group["items"])
    return {"evaluated_users": len(recalls), "CandidateRecall@1000": float(np.mean(recalls) if recalls else 0),
            "CandidateHitRate@1000": float(np.mean(hits) if hits else 0), "CatalogCoverage@1000": len(covered) / len(item_ids)}

train_groups = build_groups(train, selection)
valid_context = {user: train.get(user, []) + selection.get(user, []) for user in valid}
test_context = {user: train.get(user, []) + selection.get(user, []) + valid.get(user, []) for user in test}
valid_groups = build_groups(valid_context, valid)
test_groups = build_groups(test_context, test)
print({"train": candidate_metrics(train_groups), "valid": candidate_metrics(valid_groups), "test": candidate_metrics(test_groups)})

## 3. Residual listwise reranker and fair evaluation

In [ ]:
train_features = np.concatenate([group["x"] for group in train_groups])
mean = train_features.mean(0)
std = train_features.std(0).clip(1e-6)
for groups in (train_groups, valid_groups, test_groups):
    for group in groups:
        group["x"] = (group["x"] - mean) / std

class Ranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 64), nn.GELU(), nn.Dropout(0.15), nn.Linear(64, 1))

    def forward(self, features):
        return self.net(features).squeeze(-1)

ranker = Ranker().to(device)
optimizer = torch.optim.AdamW(ranker.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
@torch.no_grad()
def evaluate(groups, ks=(12, 50, 100), target_filter=None):
    ranker.eval()
    values = {name: {k: [] for k in ks} for name in ("map", "recall", "hit", "ndcg")}
    fallback_users = 0

    for group in groups:
        truth = {item for item in group["truth"] if target_filter is None or target_filter(group, item)}
        if not truth:
            continue
        scores = ranker(torch.tensor(group["x"], device=device))
        order = torch.argsort(scores, descending=True).cpu().numpy()
        ranked = group["items"][order].tolist()
        fallback_users += int(group["fallback"])
        hits = [int(item in truth) for item in ranked]

        for k in ks:
            hits_k = hits[:k]
            ap = sum(sum(hits_k[:pos + 1]) / (pos + 1) * hit for pos, hit in enumerate(hits_k))
            dcg = sum(hit / np.log2(pos + 2) for pos, hit in enumerate(hits_k))
            idcg = sum(1 / np.log2(pos + 2) for pos in range(min(k, len(truth))))
            values["map"][k].append(ap / min(k, len(truth)))
            values["recall"][k].append(sum(hits_k) / len(truth))
            values["hit"][k].append(float(any(hits_k)))
            values["ndcg"][k].append(dcg / idcg)

    result = {"evaluated_users": len(values["map"][ks[0]]), "fallback_popularity_users": fallback_users}
    for k in ks:
        result[f"MAP@{k}"] = float(np.mean(values["map"][k]) if values["map"][k] else 0)
        result[f"Recall@{k}"] = float(np.mean(values["recall"][k]) if values["recall"][k] else 0)
        result[f"HitRate@{k}"] = float(np.mean(values["hit"][k]) if values["hit"][k] else 0)
        result[f"NDCG@{k}"] = float(np.mean(values["ndcg"][k]) if values["ndcg"][k] else 0)
    return result

In [ ]:
epochs = int(os.getenv("HM_RERANK_EPOCHS", 4 if PROFILE == "quick" else 15))
best_ndcg, history = -1, []

for epoch in range(1, epochs + 1):
    ranker.train()
    random.shuffle(train_groups)
    losses = []
    for group in train_groups:
        labels = torch.tensor([int(item in group["truth"]) for item in group["items"]], dtype=torch.float32, device=device)
        if not labels.any():
            continue
        scores = ranker(torch.tensor(group["x"], device=device))
        loss = -(torch.log_softmax(scores, 0) * labels / labels.sum()).sum()
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(ranker.parameters(), 5)
        optimizer.step()
        losses.append(loss.item())

    metrics = evaluate(valid_groups)
    row = {"epoch": epoch, "loss": float(np.mean(losses) if losses else 0), **metrics}
    history.append(row)
    print(row)
    if metrics["NDCG@12"] > best_ndcg:
        best_ndcg = metrics["NDCG@12"]
        torch.save({"state": ranker.state_dict(), "valid": metrics, "features": ["tower_score", "log_popularity", "content_similarity", "from_tower", "from_popularity", "from_repeat", "from_content"]}, OUTPUT_DIR / "best_reranker.pt")

ranker.load_state_dict(torch.load(OUTPUT_DIR / "best_reranker.pt", map_location=device)["state"])
cold = lambda group, item: item not in train_item_set
repeat = lambda group, item: item in group["history"]
explore = lambda group, item: item not in group["history"]
result = {
    "selection_split": "valid",
    "candidate_valid": candidate_metrics(valid_groups),
    "candidate_test": candidate_metrics(test_groups),
    "candidate_test_strict_cold_items": candidate_metrics(test_groups, target_filter=cold),
    "candidate_test_repeat_items": candidate_metrics(test_groups, target_filter=repeat),
    "candidate_test_explore_items": candidate_metrics(test_groups, target_filter=explore),
    "test_overall": evaluate(test_groups),
    "test_strict_cold_items": evaluate(test_groups, target_filter=cold),
    "test_repeat_items": evaluate(test_groups, target_filter=repeat),
    "test_explore_items": evaluate(test_groups, target_filter=explore),
}
(OUTPUT_DIR / "reranker_history.json").write_text(json.dumps(history, indent=2))
(OUTPUT_DIR / "reranker_metrics.json").write_text(json.dumps(result, indent=2))
display(result)

**Báo cáo:** `candidate_*` đo khả năng giữ ground truth trong Top-1000; `test_*` đo thứ hạng sau rerank. Cold = chưa xuất hiện trong train, repeat/explore được tính theo lịch sử có sẵn tại thời điểm test.